# 🌍 Earth Field Analysis
## Layer 6 – Resonance Field / Schumann

| Layer | Name | Status |
|-------|------|--------|
| 0 | External Cosmic Drivers | ✅ |
| 1 | Planetary Body | ✅ |
| 2 | Surface / Oceans / Land | ✅ |
| 3 | Atmosphere / Weather / Thunderstorms | ✅ |
| 4 | Ionosphere | ✅ |
| 5 | Global Electric Circuit | ✅ |
| **6** | **Resonance Field / Schumann** | **← this layer** |
| 7 | Earth Field State Engine | ⬜ |
| 8 | Research / Hypotheses | ⬜ |

> **Core idea:** Schumann resonance is not just a frequency, but an expression of:
> - global lightning activity (Layer 3)
> - the Earth-Ionosphere Cavity (Layer 4)
> - altered propagation conditions (Layer 4 + 5)
> - coupled system states (all layers)
>
> **Central method (from Layer 4):** Comparison `geometric_delta` ↔ `expected_real_delta` → diagnostic mechanism for non-geometric factors.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

print(f'Analysis date: {datetime.datetime.utcnow().isoformat()}Z')

# Load all previous layers
context = {}
for n in [0, 1, 2, 3, 4, 5]:
    try:
        with open(f'layer{n}_test_state.json', encoding='utf-8') as f:
            context[n] = json.load(f)
        print(f'  Layer {n}: {context[n]["level"].upper():8}  Score={context[n]["score"]}')
    except FileNotFoundError:
        print(f'  Layer {n}: not found')
        context[n] = None

L0, L1, L2, L3, L4, L5 = (context.get(n) for n in range(6))

# Key values from upstream layers
# L3: Generator (lightning, CAPE)
l3_thunder_score = L3.get('components',{}).get('Thunderstorm Activity',{}).get('score') if L3 else None
l3_cape          = L3.get('raw_values',{}).get('CAPE_mean_Jkg') if L3 else None
l3_n_thunder     = L3.get('raw_values',{}).get('thunder_points_WMO', 0) if L3 else 0
l3_schumann_pot  = L3.get('raw_values',{}).get('schumann_potential') if L3 else None
l3_messpunkte    = L3.get('raw_values',{}).get('messpunkte', []) if L3 else []

# L4: Cavity geometry
l4_cavity_h      = L4.get('resonance_system',{}).get('cavity_height_km', 80.0) if L4 else 80.0
l4_cavity_delta  = L4.get('resonance_system',{}).get('cavity_delta_km', 0.0) if L4 else 0.0
l4_geom_freq     = L4.get('resonance_system',{}).get('schumann_frequencies_Hz',{}) if L4 else {}
l4_geom_delta_mhz= L4.get('resonance_system',{}).get('schumann_delta_mHz',{}) if L4 else {}
l4_day_night     = L4.get('resonance_system',{}).get('day_night', 'unknown') if L4 else 'unknown'
l4_kp            = L4.get('raw_values',{}).get('Kp_current') if L4 else None
l4_xray_class    = L4.get('raw_values',{}).get('xray_class') if L4 else None
l4_ioniz_score   = L4.get('components',{}).get('Ionization Level (F10.7)',{}).get('score') if L4 else None

# L5: GEC state
l5_v_iono        = L5.get('gec_state',{}).get('V_ionosphere_kV', 300.0) if L5 else 300.0
l5_delta_v_pct   = L5.get('gec_state',{}).get('delta_V_pct', 0.0) if L5 else 0.0
l5_generator     = L5.get('gec_state',{}).get('generator_strength', 1.0) if L5 else 1.0
l5_score         = L5.get('score') if L5 else None

# UT hour for diurnal pattern (Schumann chimneys)
utc_hour = datetime.datetime.utcnow().hour

print(f'\n  L3 Generator:  thunder={l3_thunder_score}  CAPE={l3_cape}  schumann_pot={l3_schumann_pot}')
print(f'  L4 Cavity:     h={l4_cavity_h} km (Δ {l4_cavity_delta:+.1f})  {l4_day_night}  Kp={l4_kp}')
print(f'  L5 GEC:        V_iono={l5_v_iono} kV (Δ {l5_delta_v_pct:+.1f}%)  generator={l5_generator}')
print(f'  UTC-Hour:    {utc_hour}h')

Analysis date: 2026-05-13T21:59:51.343124Z
  Layer 0: CALM      Score=0.1523
  Layer 1: MODERATE  Score=0.3572
  Layer 2: MODERATE  Score=0.3518
  Layer 3: QUIET     Score=0.1334
  Layer 4: QUIET     Score=0.281
  Layer 5: QUIET     Score=0.229

  L3 Generator:  thunder=0.1606  CAPE=571.7  schumann_pot=0.1867
  L4 Cavity:     h=86.2 km (Δ +6.2)  night  Kp=1.0
  L5 GEC:        V_iono=305.4 kV (Δ +1.8%)  generator=1.018
  UTC-Hour:    21h


---
## 1. Empirical Reference vs. Geometric Model

Schumann resonance has two different reference frames — the distinction is central:

| Mode | Empirically observed | Geometric model (h=80 km) | Difference |
|-------|---------------------|-------------------------------|-----------|
| SR-1  | **7.83 Hz** (Q≈4–5) | 10.46 Hz | −2.63 Hz |
| SR-2  | **14.3 Hz** | 18.12 Hz | −3.82 Hz |
| SR-3  | **20.8 Hz** | 25.62 Hz | −4.82 Hz |
| SR-4  | **27.3 Hz** | 33.08 Hz | −5.78 Hz |
| SR-5  | **33.8 Hz** | 40.51 Hz | −6.71 Hz |

**Why the difference?** The idealized cavity model neglects:
- finite conductivity of the ionosphere (D-layer)
- damping due to ionospheric losses
- effective propagation velocity < c
- day/night asymmetry

**→ Consequence:** Layer 6 uses the **empirical reference values** as the physical basis, not the geometric values from Layer 4.

**→ Diagnostic mechanism:** The geometric Δ from Layer 4 (~+16 mHz per 10 km) is the upper bound of the pure cavity effect. Real frequency shifts must be compared with this Δ to identify non-geometric drivers.

In [2]:
# ============================================================
# EMPIRICAL SCHUMANN REFERENCE VALUES
# Source: Nickolaenko & Hayakawa (2014), long-term average
# ============================================================

SR_REF = {
    1: {'freq_Hz': 7.83,  'amplitude_pT': 1.00, 'Q_factor': 4.5},
    2: {'freq_Hz': 14.3,  'amplitude_pT': 0.45, 'Q_factor': 4.8},
    3: {'freq_Hz': 20.8,  'amplitude_pT': 0.30, 'Q_factor': 5.0},
    4: {'freq_Hz': 27.3,  'amplitude_pT': 0.20, 'Q_factor': 5.2},
    5: {'freq_Hz': 33.8,  'amplitude_pT': 0.13, 'Q_factor': 5.4},
}

# Three Schumann chimneys (main global thunderstorm centres)
CHIMNEYS = {
    'Asia/Maritime':  {'lon_center': 100, 'peak_UT': 8,  'color': '#F2A623'},
    'Africa':          {'lon_center': 25,  'peak_UT': 14, 'color': '#E85D24'},
    'Americas':         {'lon_center': -75, 'peak_UT': 20, 'color': '#534AB7'},
}

print('SCHUMANN REFERENCE VALUES (empirical, long-term mean)')
print('=' * 62)
print(f'  {"Mode":<6} {"Freq [Hz]":<11} {"Amp [pT]":<11} {"Q-Factor":<10}')
print('-' * 62)
for n, ref in SR_REF.items():
    print(f'  SR-{n}   {ref["freq_Hz"]:<11.2f} {ref["amplitude_pT"]:<11.2f} {ref["Q_factor"]:<10.1f}')
print('=' * 62)
print('\nDiurnal chimneys (main thunderstorm centres):')
for name, c in CHIMNEYS.items():
    active = abs(((utc_hour - c['peak_UT']) + 12) % 24 - 12) < 4
    print(f'  {name:<18} Peak UTC {c["peak_UT"]:>2}h   {"ACTIVE" if active else "..."}  ')

SCHUMANN REFERENCE VALUES (empirical, long-term mean)
  Mode   Freq [Hz]   Amp [pT]    Q-Factor  
--------------------------------------------------------------
  SR-1   7.83        1.00        4.5       
  SR-2   14.30       0.45        4.8       
  SR-3   20.80       0.30        5.0       
  SR-4   27.30       0.20        5.2       
  SR-5   33.80       0.13        5.4       

Diurnal chimneys (main thunderstorm centres):
  Asia/Maritime      Peak UTC  8h   ...  
  Africa             Peak UTC 14h   ...  
  Americas           Peak UTC 20h   ACTIVE  


---
## 2. Modulation Model: Expected Real Schumann Values

In [3]:
# ============================================================
# MODULATION MODEL
# Expected Schumann values = Reference × modulators from L3/L4/L5
# ============================================================

# === FREQUENCY MODULATION ===
# From Layer 4: geometric delta (very small, ~16-51 mHz per 10 km)
# Plus real cavity effect (day/night, ionospheric losses)
geom_delta_mhz = {n: l4_geom_delta_mhz.get(f'SR_{n}', 0.0) for n in [1,2,3,4]}

# Day/night effect (empirically observed, ~50-100 mHz shift)
# Day: lower SR (smaller effective cavity height), Night: higher SR
day_night_shift_mhz = -50 if l4_day_night == 'day' else +50 if l4_day_night == 'night' else 0

# Ionospheric disturbance (Kp, X-Ray) → frequency shift
kp_val = l4_kp if l4_kp is not None else 2.0
kp_shift_mhz = -kp_val * 5  # higher Kp → lower frequencies
xray_shift_mhz = -30 if l4_xray_class in ['M', 'X'] else 0

# === AMPLITUDE MODULATION ===
# From Layer 3: thunderstorm activity (generator)
# From Layer 5: GEC potential (~generator strength)
amp_mod = (l5_generator or 1.0) * (1.0 + (l3_thunder_score or 0.15) * 0.5)

# Diurnal variation: ~30% variation depending on chimney
# Maximum when a chimney is at its peak
chimney_activity = 0.0
active_chimneys = []
for name, c in CHIMNEYS.items():
    distance = abs(((utc_hour - c['peak_UT']) + 12) % 24 - 12)
    activity = max(0, 1 - distance / 6)  # Full contribution if distance = 0; zero if distance ≥ 6.
    chimney_activity += activity
    if activity > 0.3:
        active_chimneys.append((name, activity))
chimney_factor = 0.85 + chimney_activity * 0.10  # 0.85–1.15
amp_mod_total = amp_mod * chimney_factor

# === Q-FACTOR MODULATION ===
# Q depends on ionospheric conductivity
# High F10.7 → lower D-Layer → higher losses → lower Q
q_mod = 1.0 - (l4_ioniz_score or 0.3) * 0.15  # up to -15% at very active sun
if l4_xray_class in ['M', 'X']:
    q_mod *= 0.7  # massive Q degradation during large flares

# === CALCULATE EXPECTED SCHUMANN VALUES ===
expected = {}
for n, ref in SR_REF.items():
    # Frequency shift (mHz → Hz)
    total_shift_mhz = (geom_delta_mhz.get(n, 0) +
                       day_night_shift_mhz +
                       kp_shift_mhz +
                       xray_shift_mhz)
    expected[n] = {
        'freq_Hz':       round(ref['freq_Hz'] + total_shift_mhz / 1000, 4),
        'freq_shift_mHz':round(total_shift_mhz, 1),
        'amplitude_pT':  round(ref['amplitude_pT'] * amp_mod_total, 4),
        'amplitude_factor': round(amp_mod_total, 3),
        'Q_factor':      round(ref['Q_factor'] * q_mod, 3),
        'Q_factor_change': round((q_mod - 1) * 100, 1),
    }

print('EXPECTED SCHUMANN VALUES (Reference × Modulators)')
print('=' * 78)
print(f'  {"Mode":<6} {"Freq [Hz]":<11} {"Δf [mHz]":<11} {"Amp [pT]":<11} {"Q":<8} {"ΔQ [%]":<8}')
print('-' * 78)
for n, e in expected.items():
    print(f'  SR-{n}   {e["freq_Hz"]:<11.4f} {e["freq_shift_mHz"]:<+11.1f} '
          f'{e["amplitude_pT"]:<11.4f} {e["Q_factor"]:<8.2f} {e["Q_factor_change"]:<+8.1f}')
print('=' * 78)

print('\nMODULATOR BREAKDOWN')
print(f'  Frequency shifts (additive):')
print(f'    geometric (Layer 4):        SR-1 {geom_delta_mhz.get(1,0):+.1f} mHz ... SR-4 {geom_delta_mhz.get(4,0):+.1f} mHz')
print(f'    Day/Night ({l4_day_night}):    {day_night_shift_mhz:+.0f} mHz')
print(f'    Kp effect (Kp={kp_val:.1f}):    {kp_shift_mhz:+.1f} mHz')
print(f'    X-Ray ({l4_xray_class}-class):  {xray_shift_mhz:+.0f} mHz')
print(f'  Amplitude factor: {amp_mod_total:.3f}')
print(f'    Generator (L5): {l5_generator:.3f}')
print(f'    Thunderstorm (L3): {(1.0 + (l3_thunder_score or 0.15) * 0.5):.3f}')
print(f'    Chimney factor: {chimney_factor:.3f}')
print(f'  Q-factor: {q_mod:.3f} (ionization + X-Ray)')
print(f'\nActive chimneys: {[c[0] for c in active_chimneys] if active_chimneys else "–"}')

EXPECTED SCHUMANN VALUES (Reference × Modulators)
  Mode   Freq [Hz]   Δf [mHz]    Amp [pT]    Q        ΔQ [%]  
------------------------------------------------------------------------------
  SR-1   7.8650      +35.0       1.0264      4.29     -4.7    
  SR-2   14.3276     +27.6       0.4619      4.57     -4.7    
  SR-3   20.8204     +20.4       0.3079      4.76     -4.7    
  SR-4   27.3132     +13.2       0.2053      4.95     -4.7    
  SR-5   33.8450     +45.0       0.1334      5.14     -4.7    

MODULATOR BREAKDOWN
  Frequency shifts (additive):
    geometric (Layer 4):        SR-1 -10.0 mHz ... SR-4 -31.8 mHz
    Day/Night (night):    +50 mHz
    Kp effect (Kp=1.0):    -5.0 mHz
    X-Ray (B-class):  +0 mHz
  Amplitude factor: 1.026
    Generator (L5): 1.018
    Thunderstorm (L3): 1.080
    Chimney factor: 0.933
  Q-factor: 0.953 (ionization + X-Ray)

Active chimneys: ['Americas']


---
## 3. Visualizations

In [4]:
# ============================================================
# SCHUMANN SPECTRUM (modelled)
# Lorentz curves for each mode, with modulation
# ============================================================

freqs = np.linspace(3, 40, 800)

def lorentzian(f, f0, A, Q):
    """Normalized Lorentz curve with Q-factor and amplitude A"""
    gamma = f0 / (2 * Q)
    return A * gamma**2 / ((f - f0)**2 + gamma**2)

spec_ref = np.zeros_like(freqs)
spec_exp = np.zeros_like(freqs)
for n, ref in SR_REF.items():
    spec_ref += lorentzian(freqs, ref['freq_Hz'], ref['amplitude_pT'], ref['Q_factor'])
    e = expected[n]
    spec_exp += lorentzian(freqs, e['freq_Hz'], e['amplitude_pT'], e['Q_factor'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=freqs, y=spec_ref, mode='lines', name='Reference (empirical)',
    line=dict(color='#888780', width=1.5, dash='dot')
))
fig.add_trace(go.Scatter(
    x=freqs, y=spec_exp, mode='lines', name='Expected (modulated)',
    line=dict(color='#F2A623', width=2.2),
    fill='tozeroy', fillcolor='rgba(242,166,35,0.13)'
))

# Mode Markers
for n, ref in SR_REF.items():
    fig.add_annotation(
        x=ref['freq_Hz'], y=ref['amplitude_pT'] * 1.1,
        text=f'SR-{n}<br>{ref["freq_Hz"]:.2f} Hz',
        showarrow=False, font=dict(size=9, color='#888780'),
    )

fig.update_layout(
    title=dict(text='Schumann Spectrum: Empirical vs. expected modulated', font=dict(size=14)),
    xaxis=dict(title='Frequency [Hz]', range=[3, 40], gridcolor='#222244'),
    yaxis=dict(title='Amplitude [pT]', gridcolor='#222244'),
    height=380, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=30, t=55, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

In [5]:
# ============================================================
# THREE-COLUMN COMPARISON PER MODE:
# Frequency | Amplitude | Q-Factor (Ref vs. Expected each)
# ============================================================

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Frequency [Hz]', 'Amplitude [pT]', 'Q-Factor'],
    horizontal_spacing=0.10
)

modes = list(SR_REF.keys())
labels = [f'SR-{n}' for n in modes]

# Frequenz
fig.add_trace(go.Bar(
    name='Reference', x=labels, y=[SR_REF[n]['freq_Hz'] for n in modes],
    marker_color='#5F5E5A', opacity=0.75, showlegend=True
), row=1, col=1)
fig.add_trace(go.Bar(
    name='Expected', x=labels, y=[expected[n]['freq_Hz'] for n in modes],
    marker_color='#F2A623', opacity=0.90,
    text=[f'{expected[n]["freq_Hz"]:.2f}' for n in modes],
    textposition='outside', textfont=dict(color='white', size=9),
    showlegend=True
), row=1, col=1)

# Amplitude
fig.add_trace(go.Bar(
    x=labels, y=[SR_REF[n]['amplitude_pT'] for n in modes],
    marker_color='#5F5E5A', opacity=0.75, showlegend=False
), row=1, col=2)
fig.add_trace(go.Bar(
    x=labels, y=[expected[n]['amplitude_pT'] for n in modes],
    marker_color='#E85D24', opacity=0.90,
    text=[f'{expected[n]["amplitude_pT"]:.2f}' for n in modes],
    textposition='outside', textfont=dict(color='white', size=9),
    showlegend=False
), row=1, col=2)

# Q-Faktor
fig.add_trace(go.Bar(
    x=labels, y=[SR_REF[n]['Q_factor'] for n in modes],
    marker_color='#5F5E5A', opacity=0.75, showlegend=False
), row=1, col=3)
fig.add_trace(go.Bar(
    x=labels, y=[expected[n]['Q_factor'] for n in modes],
    marker_color='#534AB7', opacity=0.90,
    text=[f'{expected[n]["Q_factor"]:.2f}' for n in modes],
    textposition='outside', textfont=dict(color='white', size=9),
    showlegend=False
), row=1, col=3)

fig.update_layout(
    title=dict(text='Schumann Modes: Empirical Reference vs. Expected Modulated Value', font=dict(size=13)),
    barmode='group',
    height=400, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=50, r=30, t=70, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

In [6]:
# ============================================================
# DIURNAL PATTERN – Schumann chimneys (UTC diurnal curve)
# ============================================================

ut_range = list(range(0, 24))

def chimney_curve(peak_ut, ut):
    """Bell curve around peak UT with half-width ~6h"""
    distance = abs(((ut - peak_ut) + 12) % 24 - 12)
    return max(0, math.exp(-(distance / 5)**2))

fig = go.Figure()
total_curve = [0.0] * 24
for name, c in CHIMNEYS.items():
    curve = [chimney_curve(c['peak_UT'], h) for h in ut_range]
    total_curve = [t + v for t, v in zip(total_curve, curve)]
    fig.add_trace(go.Scatter(
        x=ut_range, y=curve,
        mode='lines', name=name,
        line=dict(color=c['color'], width=2),
        fill='tozeroy', fillcolor=c['color'].replace('#', 'rgba(') if False else None,
        opacity=0.7,
    ))
fig.add_trace(go.Scatter(
    x=ut_range, y=total_curve,
    mode='lines', name='Global Activity',
    line=dict(color='white', width=2.5, dash='dash')
))

# Mark current UTC hour
fig.add_vline(x=utc_hour, line_color='#F2A623', line_width=2,
              annotation_text=f'now: {utc_hour}h UTC',
              annotation_font=dict(color='#F2A623', size=10))

fig.update_layout(
    title=dict(text='Schumann Chimneys: Diurnal pattern (UTC) of global lightning activity', font=dict(size=14)),
    xaxis=dict(title='UTC hour', tickmode='linear', tick0=0, dtick=2,
               range=[0, 23], gridcolor='#222244'),
    yaxis=dict(title='Relative excitation strength', gridcolor='#222244'),
    height=350, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=30, t=55, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

In [7]:
# ============================================================
# DELTA DIAGNOSIS: geometric (L4) vs. non-geometric (modulatory)
# Core question: What fraction of the expected frequency shift
# comes from pure cavity geometry, and which from other drivers?
# ============================================================

modes_d = list(SR_REF.keys())
geom_components   = [geom_delta_mhz.get(n, 0)   for n in modes_d]
daynight_components = [day_night_shift_mhz       for _ in modes_d]
kp_components     = [kp_shift_mhz                for _ in modes_d]
xray_components   = [xray_shift_mhz              for _ in modes_d]
totals            = [expected[n]['freq_shift_mHz'] for n in modes_d]

fig = go.Figure()
fig.add_trace(go.Bar(name='Geometric (L4)',
    x=[f'SR-{n}' for n in modes_d], y=geom_components,
    marker_color='#888780', opacity=0.85))
fig.add_trace(go.Bar(name=f'Day/Night ({l4_day_night})',
    x=[f'SR-{n}' for n in modes_d], y=daynight_components,
    marker_color='#378ADD', opacity=0.85))
fig.add_trace(go.Bar(name=f'Kp ({kp_val:.1f})',
    x=[f'SR-{n}' for n in modes_d], y=kp_components,
    marker_color='#7F77DD', opacity=0.85))
fig.add_trace(go.Bar(name=f'X-Ray ({l4_xray_class})',
    x=[f'SR-{n}' for n in modes_d], y=xray_components,
    marker_color='#E85D24', opacity=0.85))

# Total as a Line
fig.add_trace(go.Scatter(
    x=[f'SR-{n}' for n in modes_d], y=totals,
    mode='lines+markers+text', name='sum',
    line=dict(color='#F2A623', width=2.5),
    marker=dict(size=10, color='#F2A623'),
    text=[f'{t:+.0f}' for t in totals],
    textposition='top center',
    textfont=dict(color='#F2A623', size=10)
))

fig.add_hline(y=0, line_color='#888780', line_width=0.8)
fig.update_layout(
    title=dict(text='Delta Diagnosis: Frequency Shift Breakdown by Driver [mHz]', font=dict(size=14)),
    barmode='relative',
    yaxis=dict(title='Frequency shift [mHz]', gridcolor='#222244'),
    height=380, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=30, t=55, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

# interpretation
geom_total_sr1     = abs(geom_components[0])
non_geom_total_sr1 = abs(daynight_components[0]) + abs(kp_components[0]) + abs(xray_components[0])
ratio = non_geom_total_sr1 / max(geom_total_sr1, 1.0)
print(f'\nSR-1 driver ratio:')
print(f'  geometric:       {geom_total_sr1:.1f} mHz')
print(f'  non-geometric:   {non_geom_total_sr1:.1f} mHz')
print(f'  ratio:           {ratio:.1f}x')
if ratio > 3:
    print('  → Non-geometric factors clearly dominant (verification of Layer-4 hypothesis)')


SR-1 driver ratio:
  geometric:       10.0 mHz
  non-geometric:   55.0 mHz
  ratio:           5.5x
  → Non-geometric factors clearly dominant (verification of Layer-4 hypothesis)


---
## 4. State Assessment & Handoff to Layer 7

In [8]:
# ============================================================
# LAYER-6-SCORE
# ============================================================

def norm(v, lo, hi):
    if v is None: return None
    return round(max(0.0, min(1.0, (v - lo) / (hi - lo))), 4)

# 1) Excitation strength (from amplitude modulator)
excitation = norm(amp_mod_total, 0.5, 1.8)

# 2) Frequency anomaly (total shift of SR-1)
freq_anomaly = norm(abs(expected[1]['freq_shift_mHz']), 0, 200)

# 3) Q-factor deviation
q_anomaly = norm(abs(expected[1]['Q_factor_change']), 0, 30)

# 4) Chimney activity
chimney_score = norm(chimney_factor, 0.85, 1.15)

# 5) Non-geometric dominance (diagnostic feature)
non_geom_dom = norm(ratio, 0, 10)

COMPONENTS = {
    'Excitation Strength (Amplitude)':  {'score': excitation,    'source': 'derived_from_L3_L5', 'dynamic': True},
    'Frequency Anomaly (SR-1)':       {'score': freq_anomaly,  'source': 'derived_from_L4',   'dynamic': True},
    'Q-Factor Deviation':            {'score': q_anomaly,     'source': 'derived_from_L4',   'dynamic': True},
    'Chimney Activity':             {'score': chimney_score, 'source': 'time_of_day',       'dynamic': True},
    'Non-geom. Dominance':           {'score': non_geom_dom,  'source': 'diagnostic',        'dynamic': True},
}

available    = {k: v['score'] for k, v in COMPONENTS.items() if v['score'] is not None}
unavailable  = [k for k, v in COMPONENTS.items() if v['score'] is None]
layer6_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)
level = ('unknown' if layer6_score is None
         else 'quiet'   if layer6_score < 0.3
         else 'moderate' if layer6_score < 0.6
         else 'active')
dominant_l6 = max(available, key=available.get) if available else 'none'

W = 70
print('=' * W)
print('LAYER 6 – RESONANCE FIELD / SCHUMANN – STATE ASSESSMENT')
print('=' * W)
for name, comp in COMPONENTS.items():
    s = comp['score']
    if s is not None:
        bar = '█' * int(s*20) + '░' * (20-int(s*20))
        print(f'  {name:<32} {bar}  {s:.3f}  [{comp["source"]}]')
    else:
        print(f'  {name:<32} {"─"*20}  n/a   [missing]')
print('-' * W)
print(f'  Score:           {layer6_score:.3f}  ({len(available)}/{len(COMPONENTS)} components)')
print(f'  Confidence:      {confidence:.0%}')
print(f'  Level:           {level.upper()}')
print(f'  Dominant:        {dominant_l6}')
print(f'  SR-1 expected:   {expected[1]["freq_Hz"]:.4f} Hz  (Δ {expected[1]["freq_shift_mHz"]:+.1f} mHz vs 7.83 Hz)')
print(f'  Amplitude:       factor {amp_mod_total:.3f}  (vs reference)')
print(f'  Q-Factor SR-1:   {expected[1]["Q_factor"]:.2f}  ({expected[1]["Q_factor_change"]:+.1f}%)')
print('=' * W)

# Radar
cats   = list(available.keys())
vals_r = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals_r + [vals_r[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(83,74,183,0.20)',
        line=dict(color='#534AB7', width=2.5), name='Layer 6'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.5]*(len(cats)+1), theta=cats+[cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1),
        mode='lines', name='Activity Threshold'
    ))
    fig.update_layout(
        title=dict(
            text=f'Layer 6 – Resonance Field | Score: {layer6_score:.3f} | {level.upper()} | Confidence: {confidence:.0%}',
            font=dict(size=12)
        ),
        polar=dict(radialaxis=dict(visible=True, range=[0,1])),
        height=450, showlegend=True,
        margin=dict(l=80, r=80, t=70, b=40)
    )
    fig.show()

LAYER 6 – RESONANCE FIELD / SCHUMANN – STATE ASSESSMENT
  Excitation Strength (Amplitude)  ████████░░░░░░░░░░░░  0.405  [derived_from_L3_L5]
  Frequency Anomaly (SR-1)         ███░░░░░░░░░░░░░░░░░  0.175  [derived_from_L4]
  Q-Factor Deviation               ███░░░░░░░░░░░░░░░░░  0.157  [derived_from_L4]
  Chimney Activity                 █████░░░░░░░░░░░░░░░  0.278  [time_of_day]
  Non-geom. Dominance              ██████████░░░░░░░░░░  0.548  [diagnostic]
----------------------------------------------------------------------
  Score:           0.312  (5/5 components)
  Confidence:      100%
  Level:           MODERATE
  Dominant:        Non-geom. Dominance
  SR-1 expected:   7.8650 Hz  (Δ +35.0 mHz vs 7.83 Hz)
  Amplitude:       factor 1.026  (vs reference)
  Q-Factor SR-1:   4.29  (-4.7%)


In [9]:
# ============================================================
# EXPORT – layer6_test_state.json
# ============================================================

_amp_str  = ('elevated excitation' if amp_mod_total > 1.2
             else 'damped excitation' if amp_mod_total < 0.85
             else 'normal excitation')
_freq_str = ('significant frequency shift' if abs(expected[1]['freq_shift_mHz']) > 100
             else 'frequency near reference')

state_summary = (
    f'Layer-6-State: {level}. '
    f'SR-1 expected {expected[1]["freq_Hz"]:.3f} Hz '
    f'(Δ {expected[1]["freq_shift_mHz"]:+.1f} mHz vs reference 7.83 Hz). '
    f'Amplitude factor {amp_mod_total:.2f} ({_amp_str}). '
    f'Q-Factor SR-1 {expected[1]["Q_factor"]:.2f} ({expected[1]["Q_factor_change"]:+.1f}%). '
    f'Active chimneys: {[c[0] for c in active_chimneys] if active_chimneys else "–"}. '
    f'Non-geometric dominance: {ratio:.1f}x. '
    f'Data completeness: {confidence:.0%}.'
)

layer6_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 6,
    'name':  'Resonance Field / Schumann',

    # Important: all values are modelled expectations, not real measurements
    'measurement_status':      'model_expected_not_observed',
    'observed_data_available': False,
    'observed': {
        'SR_1_freq_Hz':    None,
        'SR_1_amplitude_pT': None,
        'Q_factor':        None,
        'note': 'Insert real measurement data here when available (e.g. AURA, NASA/MEPAS)'
    },

    'score':      layer6_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} dynamic components',
    'dominant_component': dominant_l6,
    'missing_components': unavailable,

    'components': {
        k: {'score': round(v['score'], 4) if v['score'] is not None else None,
            'source': v['source'], 'dynamic': v['dynamic']}
        for k, v in COMPONENTS.items()
    },

    # Empirical Reference (Always Available)
    'empirical_reference': {
        f'SR_{n}': {**v, 'unit': {'freq_Hz': 'Hz', 'amplitude_pT': 'pT', 'Q_factor': '–'}}
        for n, v in SR_REF.items()
    },

    # Modulated Expectation per Mode
    'expected_modulated': {
        f'SR_{n}': v for n, v in expected.items()
    },

    # Modulator Breakdown (for Layer-7 Engine)
    'modulators': {
        'frequency_shifts_mHz': {
            'geometric_from_L4': geom_delta_mhz,
            'day_night':         day_night_shift_mhz,
            'kp_effect':         kp_shift_mhz,
            'xray_effect':       xray_shift_mhz,
        },
        'amplitude_factor': {
            'total':           round(amp_mod_total, 4),
            'gec_generator':   round(l5_generator, 3),
            'thunder_l3':      round(1.0 + (l3_thunder_score or 0.15) * 0.5, 3),
            'chimney_factor':  round(chimney_factor, 3),
        },
        'q_factor_modulation': {
            'total':         round(q_mod, 4),
            'ionization':    round(1.0 - (l4_ioniz_score or 0.3) * 0.15, 4),
            'xray_penalty':  0.7 if l4_xray_class in ['M','X'] else 1.0,
        },
    },

    # Diurnal pattern (chimneys)
    'diurnal_pattern': {
        'utc_hour': utc_hour,
        'active_chimneys': [{'name': n, 'activity': round(a, 3)} for n, a in active_chimneys],
        'global_activity': round(chimney_activity, 3),
    },

    # Delta diagnosis – core output for Layer 7
    # Which non-geometric factor is currently truly dominant?
    'dominant_physical_driver': (
        'kp_geomagnetic'        if abs(kp_shift_mhz) > abs(day_night_shift_mhz) and abs(kp_shift_mhz) > 30
        else 'solar_xray_flare' if abs(xray_shift_mhz) > 20
        else 'diurnal_chimney_distribution'
    ),
    'delta_analysis': {
        'geometric_total_SR1_mHz':      round(geom_total_sr1, 2),
        'non_geometric_total_SR1_mHz':  round(non_geom_total_sr1, 2),
        'ratio_non_geom_to_geom':       round(ratio, 2),
        'interpretation': (
            'non_geometric_dominant' if ratio > 3
            else 'mixed' if ratio > 1
            else 'geometric_dominant'
        ),
        'non_geometric_dominance_level': (
            'strong_confirmed'  if ratio > 5
            else 'weak_confirmed' if ratio > 3
            else 'marginal'       if ratio > 2
            else 'absent'
        ),
        'non_geometric_dominance_margin': round(ratio - 3.0, 3),
        'note': (
            'Non-geometric factors (day/night, Kp, X-Ray) shift '
            'the resonance significantly more than the pure cavity height (Layer 4).'
        ) if ratio > 3 else 'Geometric and non-geometric effects balanced.'
    },

    'flags': {
        'amplitude_elevated':    bool(amp_mod_total > 1.2),
        'amplitude_suppressed':  bool(amp_mod_total < 0.85),
        'frequency_anomaly':     bool(abs(expected[1]['freq_shift_mHz']) > 100),
        'q_factor_degraded':     bool(expected[1]['Q_factor_change'] < -10),
        'chimney_active':        bool(len(active_chimneys) > 0),
        'non_geometric_dominant': bool(ratio > 3),
    },

    'thresholds': {
        'amp_elevated':       1.2,
        'amp_suppressed':     0.85,
        'freq_anomaly_mHz':   100.0,
        'q_degraded_pct':     -10.0,
        'non_geom_ratio':     3.0,
    },
    'level_thresholds': {
        'quiet':    [0.0,  0.30],
        'moderate': [0.30, 0.60],
        'active':   [0.60, 0.80],
        'strong':   [0.80, 1.0],
        'note': 'Score 0.30 is lower bound of moderate; values near threshold should be interpreted as quiet_to_moderate'
    },

    'downstream_expectation': {
        'layer7_engine': (
            f'L6-Score {layer6_score:.3f} ({level}) – '
            f'SR-1 {expected[1]["freq_Hz"]:.2f} Hz | '
            f'Amp {amp_mod_total:.2f}x | '
            f'non_geom_dom={ratio:.1f}x'
        ),
        'layer8_research': (
            'Hypothesis: If real measurements confirm non_geometric_dominance, '
            'then day/night structure, conductivity/damping '
            'and source distribution (chimneys) are currently dominant. '
            'Kp and solar flares become relevant only during active disturbances.'
        ),
    },

    'layer_context': {
        'L0': {'score': L0['score'], 'level': L0['level']} if L0 else None,
        'L1': {'score': L1['score'], 'level': L1['level']} if L1 else None,
        'L2': {'score': L2['score'], 'level': L2['level']} if L2 else None,
        'L3': {'score': L3['score'], 'level': L3['level'],
               'thunder_score': l3_thunder_score,
               'CAPE_mean': l3_cape,
               'n_thunder': l3_n_thunder} if L3 else None,
        'L4': {'score': L4['score'], 'level': L4['level'],
               'cavity_height_km': l4_cavity_h,
               'cavity_delta_km': l4_cavity_delta,
               'day_night': l4_day_night,
               'cavity_delta_insight': L4.get('resonance_system',{}).get('cavity_delta_insight')} if L4 else None,
        'L5': {'score': L5['score'], 'level': L5['level'],
               'V_iono_kV': l5_v_iono,
               'delta_V_pct': l5_delta_v_pct,
               'generator_strength': l5_generator} if L5 else None,
    },

    'state_summary': state_summary,
}

# numpy cleanup
def _to_python(obj):
    if isinstance(obj, dict):  return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj
layer6_state = _to_python(layer6_state)

with open('layer6_test_state.json', 'w', encoding='utf-8') as f:
    json.dump(layer6_state, f, indent=2, ensure_ascii=False)

print('layer6_test_state.json saved')
print(json.dumps(layer6_state, indent=2, ensure_ascii=False))


layer6_test_state.json saved
{
  "timestamp": "2026-05-13T21:59:52.898654Z",
  "layer": 6,
  "name": "Resonance Field / Schumann",
  "measurement_status": "model_expected_not_observed",
  "observed_data_available": false,
  "observed": {
    "SR_1_freq_Hz": null,
    "SR_1_amplitude_pT": null,
    "Q_factor": null,
    "note": "Insert real measurement data here when available (e.g. AURA, NASA/MEPAS)"
  },
  "score": 0.3124,
  "level": "moderate",
  "confidence": 1.0,
  "score_basis": "5/5 dynamic components",
  "dominant_component": "Non-geom. Dominance",
  "missing_components": [],
  "components": {
    "Excitation Strength (Amplitude)": {
      "score": 0.4049,
      "source": "derived_from_L3_L5",
      "dynamic": true
    },
    "Frequency Anomaly (SR-1)": {
      "score": 0.175,
      "source": "derived_from_L4",
      "dynamic": true
    },
    "Q-Factor Deviation": {
      "score": 0.1567,
      "source": "derived_from_L4",
      "dynamic": true
    },
    "Chimney Activity": {


---
## Summary Layer 6

| Aspekt | Inhalt |
|--------|--------|
| **Role** | Observable EM pattern of the overall system – Schumann resonance |
| **Empirical basis** | SR-1 7.83 Hz, SR-2 14.3 Hz, SR-3 20.8 Hz, SR-4 27.3 Hz, SR-5 33.8 Hz |
| **Frequency modulators** | geometric (L4) + day/night + Kp + X-Ray |
| **Amplitude modulators** | generator (L5), thunderstorms (L3), chimney (UTC) |
| **Q-factor modulator** | Ionospheric conductivity (L4) |
| **Diurnal pattern** | 3 chimneys: Asia (8h), Africa (14h), Americas (20h UTC) |
| **Delta diagnosis** | `non_geometric_dominance = real_Δ / geometric_Δ` |
| **→ Layer 7** | `expected_modulated`, `modulators`, `delta_analysis` as engine input |
| **→ Layer 8** | Diagnostic hypothesis as research trigger |
| **Output** | `layer6_test_state.json` with complete resonance pattern |

> **Next step:** `layer7_Earth_Field.ipynb` – Earth Field State Engine